In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler  
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import traceback

# Configurar o TensorFlow para usar menos memória
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# --- Função para carregar dados ---
def carregar_dados_ticker(ticker, pasta_dados_preco):
    """Carrega dados de preço e cria features de preço anterior (T) e target (T+1)."""
    # Ajustar nome do arquivo (substituir ponto por underscore)
    ticker_nome = ticker.replace('.', '_')
    nome_arquivo = f"{ticker_nome}_historico.csv"
    caminho_arquivo = os.path.join(pasta_dados_preco, nome_arquivo)

    if not os.path.exists(caminho_arquivo):
        print(f"Erro: Arquivo não encontrado para {ticker} em {caminho_arquivo}")
        return None, None, None

    try:
        print(f"Carregando dados de preço de: {caminho_arquivo}")
        df = pd.read_csv(caminho_arquivo, index_col=0)
        df.index = pd.to_datetime(df.index)
        
        print(f"Colunas originais: {df.columns.tolist()}")
        
        # Verificar se existe a coluna Close
        if 'Close' not in df.columns:
            print(f"Erro: Coluna 'Close' não encontrada para {ticker}")
            return None, None, None
        
        # Criar feature (preço em T) e target (preço em T+1)
        df['Close_Feature'] = df['Close']
        df['Close_Target'] = df['Close'].shift(-1)  # Preço do próximo dia
        
        # Remover última linha (não tem target)
        df = df[:-1].copy()
        
        feature_cols = ['Close_Feature']
        target_col = 'Close_Target'
        
        # Remover NaNs
        df = df[feature_cols + [target_col]].copy()
        df.dropna(inplace=True)
        
        print(f"Features identificadas para {ticker}: {feature_cols}")
        print(f"Target identificado para {ticker}: {target_col}")
        print(f"Dimensões finais do DataFrame para {ticker}: {df.shape}")
        
        if df.empty:
            print(f"Erro: DataFrame ficou vazio para {ticker} após limpeza.")
            return None, None, None

        return df, feature_cols, target_col

    except Exception as e:
        print(f"Erro CRÍTICO ao carregar/processar {caminho_arquivo}: {e}")
        traceback.print_exc()
        return None, None, None

# --- Função para criar janelas multivariadas com diferenciação ---
def criar_janelas_multivariadas_diff(features_array, target_array, janela, add_diff=True):
    """Cria janelas multivariadas com diferenciação para melhorar previsões"""
    X, y = [], []
    
    if add_diff and features_array.shape[1] > 0:
        # Adicionar colunas de diferenciação para cada feature
        diffs = np.diff(features_array, axis=0)
        # Concatenar com zeros no início para manter dimensões
        zeros_row = np.zeros((1, diffs.shape[1]))
        diffs_padded = np.vstack([zeros_row, diffs])
        
        # Combinar features originais com suas diferenciações
        features_array = np.concatenate([features_array, diffs_padded], axis=1)
    
    if len(features_array) <= janela:
        print(f"Aviso: dados insuficientes ({len(features_array)}) para janela ({janela}).")
        return np.array(X), np.array(y)
    
    for i in range(len(features_array) - janela):
        X.append(features_array[i:(i + janela), :])
        y.append(target_array[i + janela])  # Target já é T+1
    
    return np.array(X), np.array(y)

# --- Modelo LSTM melhorado com Residual Connections ---
def build_model_improved(input_shape, dropout_rate=0.2):
    """Constrói modelo LSTM com conexões residuais para evitar previsões travadas"""
    inputs = layers.Input(shape=input_shape)
    
    # Camada LSTM 1
    lstm1 = layers.LSTM(100, return_sequences=True)(inputs)
    lstm1 = layers.Dropout(dropout_rate)(lstm1)
    
    # Camada LSTM 2 com conexão residual
    lstm2 = layers.LSTM(100, return_sequences=False)(lstm1)
    lstm2 = layers.Dropout(dropout_rate)(lstm2)
    
    # Atenção à última dimensão da entrada para criar conexão residual
    input_flattened = layers.Flatten()(inputs)
    input_dense = layers.Dense(100)(input_flattened)
    
    # Combinar saída do LSTM e entrada (conexão residual)
    combined = layers.Add()([lstm2, input_dense])
    combined = layers.Activation('relu')(combined)
    
    # Camadas densas finais
    dense1 = layers.Dense(64, activation='relu')(combined)
    dense1 = layers.Dropout(dropout_rate/2)(dense1)
    
    # Saída
    outputs = layers.Dense(1)(dense1)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mean_squared_error'
    )
    return model

# --- Função principal para treinar e avaliar o modelo ---
def treinar_avaliar_modelo(ticker, janela, df_full, feature_cols, target_col, 
                           resultados_dir="Resultados_Preco"):
    """Treina e avalia o modelo LSTM para um ticker e janela usando apenas dados de preço"""
    print(f"\n--- Iniciando Experimento: {ticker}, Janela {janela} ---")
    
    # Criar estrutura de pastas organizada por ticker
    ticker_dir = os.path.join(resultados_dir, ticker)
    graficos_dir = os.path.join(ticker_dir, "Graficos")
    metricas_dir = os.path.join(ticker_dir, "Metricas")
    previsoes_dir = os.path.join(ticker_dir, "Previsoes")
    hiperparametros_dir = os.path.join(ticker_dir, "Hiperparametros")
    
    # Criar todas as pastas
    for pasta in [graficos_dir, metricas_dir, previsoes_dir, hiperparametros_dir]:
        os.makedirs(pasta, exist_ok=True)
    
    # Definir caminhos dos arquivos
    metrics_path = os.path.join(metricas_dir, f"{ticker}_Janela_{janela}_metrics.csv")
    grafico_path = os.path.join(graficos_dir, f"{ticker}_Janela_{janela}_grafico_teste_final.png")
    previsoes_path = os.path.join(previsoes_dir, f"{ticker}_Janela_{janela}_previsoes_teste_final.csv")
    hiperparametros_path = os.path.join(hiperparametros_dir, f"{ticker}_Janela_{janela}_hiperparametros.csv")
    
    if os.path.exists(metrics_path):
        print(f"Resultados já existem para {ticker}, Janela {janela}. Pulando...")
        return
    
    # 1. Separar Dados Treino/Validação (2020-2022) e Teste (2023)
    start_date_val = "2020-01-01"
    end_date_val = "2022-12-31"
    start_date_test = "2023-01-01"
    end_date_test = "2023-12-31"
    
    try:
        df_val_train = df_full.loc[start_date_val:end_date_val].copy()
        df_test_final = df_full.loc[start_date_test:end_date_test].copy()
    except KeyError as e:
        print(f"Erro ao dividir dados por data para {ticker}: {e}.")
        if not df_full.empty: 
            print(f"Datas disponíveis: {df_full.index.min()} a {df_full.index.max()}")
        return
    
    if df_val_train.empty or df_test_final.empty:
        print(f"Erro: Período treino/val ou teste vazio para {ticker}.")
        return
    
    # 2. Preparar dados
    features_val_train = df_val_train[feature_cols].values
    target_val_train = df_val_train[target_col].values
    
    features_test = df_test_final[feature_cols].values
    target_test = df_test_final[target_col].values
    
    # 3. Escalonamento com StandardScaler
    scaler_features = StandardScaler()
    scaler_target = StandardScaler()
    
    # Ajustar escaladores apenas nos dados de treino
    scaled_features_val_train = scaler_features.fit_transform(features_val_train)
    target_val_train_reshaped = target_val_train.reshape(-1, 1)
    scaled_target_val_train = scaler_target.fit_transform(target_val_train_reshaped).flatten()
    
    # Transformar dados de teste
    scaled_features_test = scaler_features.transform(features_test)
    target_test_reshaped = target_test.reshape(-1, 1)
    scaled_target_test = scaler_target.transform(target_test_reshaped).flatten()
    
    # 4. Criar janelas com diferenciação
    X_train, y_train = criar_janelas_multivariadas_diff(
        scaled_features_val_train, 
        scaled_target_val_train, 
        janela,
        add_diff=True  # Adicionar colunas de diferenciação
    )
    
    X_test, y_test = criar_janelas_multivariadas_diff(
        scaled_features_test, 
        scaled_target_test, 
        janela,
        add_diff=True  # Adicionar colunas de diferenciação
    )
    
    if len(X_train) == 0 or len(X_test) == 0:
        print(f"Erro: Não foi possível criar janelas para {ticker}.")
        return
    
    # 5. Configurar validação
    val_split_idx = int(0.8 * len(X_train))
    X_train_final, y_train_final = X_train[:val_split_idx], y_train[:val_split_idx]
    X_val, y_val = X_train[val_split_idx:], y_train[val_split_idx:]
    
    # 6. Construir e treinar modelo
    input_shape = (janela, X_train.shape[2])
    dropout_rate = 0.2
    learning_rate = 0.001
    batch_size = 32
    epochs = 200
    patience = 20
    
    model = build_model_improved(input_shape, dropout_rate=dropout_rate)
    
    # Callbacks para melhorar treinamento
    early_stopping = EarlyStopping(
        monitor='val_loss', 
        patience=patience, 
        restore_best_weights=True, 
        verbose=1
    )
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_lr=1e-5,
        verbose=1
    )
    
    # Treinar modelo
    history = model.fit(
        X_train_final, y_train_final,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stopping, reduce_lr],
        verbose=1
    )
    
    # Salvar hiperparâmetros
    hiperparametros_df = pd.DataFrame([{
        'Ticker': ticker,
        'Janela': janela,
        'LSTM_Units_1': 100,
        'LSTM_Units_2': 100,
        'Dense_Units': 64,
        'Dropout_Rate': dropout_rate,
        'Learning_Rate': learning_rate,
        'Batch_Size': batch_size,
        'Epochs_Max': epochs,
        'Epochs_Trained': len(history.history['loss']),
        'Early_Stopping_Patience': patience,
        'Optimizer': 'Adam',
        'Loss_Function': 'MSE',
        'Scaler': 'StandardScaler',
        'Add_Diff_Features': True
    }])
    hiperparametros_df.to_csv(hiperparametros_path, index=False)
    
    # 7. Avaliar no conjunto de teste
    predictions_scaled = model.predict(X_test)
    predictions = scaler_target.inverse_transform(predictions_scaled).flatten()
    actual = scaler_target.inverse_transform(y_test.reshape(-1, 1)).flatten()
    
    # 8. Calcular métricas
    mae = mean_absolute_error(actual, predictions)
    mse = mean_squared_error(actual, predictions)
    rmse = np.sqrt(mse)
    r2 = r2_score(actual, predictions)
    
    # 9. Verificar variação das previsões (para detectar previsões travadas)
    var_real = np.std(actual) / np.mean(actual) if np.mean(actual) != 0 else 0
    var_pred = np.std(predictions) / np.mean(predictions) if np.mean(predictions) != 0 else 0
    prop_var = var_pred / var_real if var_real != 0 else 0
    travado = "Sim" if prop_var < 0.3 else "Não"
    
    print("\nResultados da Avaliação:")
    print(f"MAE: {mae:.4f}, MSE: {mse:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")
    print(f"Variação Real: {var_real:.4f}, Variação Prevista: {var_pred:.4f}")
    print(f"Previsões travadas? {travado}")
    
    # 10. Salvar resultados
    # Métricas
    metrics_df = pd.DataFrame([{
        'Ticker': ticker,
        'Janela': janela,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
        'Var_Real': var_real,
        'Var_Pred': var_pred,
        'Prop_Var': prop_var,
        'Travado': travado
    }])
    metrics_df.to_csv(metrics_path, index=False)
    
    # Datas para as previsões
    datas_teste = df_test_final.index[janela:janela+len(actual)]
    
    if len(datas_teste) == len(actual):
        # Salvar previsões
        previsoes_df = pd.DataFrame({
            'Data': datas_teste,
            'Preço Real': actual,
            'Preço Previsto': predictions
        })
        previsoes_df.to_csv(previsoes_path, index=False)
        
        # Gerar gráfico
        plt.figure(figsize=(14, 7))
        plt.plot(previsoes_df['Data'], previsoes_df['Preço Real'], label='Real', color='blue', linewidth=1.5)
        plt.plot(previsoes_df['Data'], previsoes_df['Preço Previsto'], label='Previsto', color='orange', linestyle='--', linewidth=1.5)
        plt.title(f'Preços Reais vs Previstos - {ticker} (Janela {janela})')
        plt.xlabel('Data')
        plt.ylabel('Preço')
        plt.legend()
        plt.grid(alpha=0.4)
        plt.tight_layout()
        plt.savefig(grafico_path)
        plt.close()
    
    print(f"--- Experimento concluído: {ticker}, Janela {janela} ---")
    return metrics_df

# --- Função para rodar experimentos em todos os tickers ---
def rodar_experimentos_preco():
    """Roda experimentos usando apenas dados de preço"""
    pasta_dados = r"C:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\CodigoExperimentos\ExperimentoFeatures\dados_preco"
    resultados_dir = r"C:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\CodigoExperimentos\ExperimentoFeatures\Resultados_Preco"
    os.makedirs(resultados_dir, exist_ok=True)
    
    # Lista de tickers
    tickers = ["BEEF3.SA", "BRFS3.SA", "CSNA3.SA", "GGBR3.SA", "JBSS3.SA", "SOJA3.SA", "SUZB3.SA", "VALE3.SA"]
    
    # Janelas a testar
    janelas = [1, 2, 3, 4, 5]
    
    resultados_consolidados = []
    
    for ticker in tickers:
        df_ticker_full, feature_cols, target_col = carregar_dados_ticker(ticker, pasta_dados)
        
        if df_ticker_full is None:
            print(f"Pulando ticker {ticker} devido a erro no carregamento.")
            continue
        
        for janela in janelas:
            try:
                metrics = treinar_avaliar_modelo(ticker, janela, df_ticker_full, feature_cols, target_col, resultados_dir)
                if metrics is not None:
                    resultados_consolidados.append(metrics)
            except Exception as e:
                print(f"Erro ao processar {ticker}, Janela {janela}: {e}")
                traceback.print_exc()
    
    # Consolidar resultados
    if resultados_consolidados:
        df_resultados = pd.concat(resultados_consolidados, ignore_index=True)
        df_resultados.to_csv(f"{resultados_dir}/resultados_consolidados.csv", index=False)
        print(f"\nResultados consolidados salvos em {resultados_dir}/resultados_consolidados.csv")

if __name__ == "__main__":
    # Desativar mensagens de aviso do TensorFlow
    tf.get_logger().setLevel('ERROR')
    
    print("Iniciando experimentos com modelos LSTM usando apenas dados de preço...")
    rodar_experimentos_preco()
    print("\nTodos experimentos concluídos!")

Iniciando experimentos com modelos LSTM usando apenas dados de preço...
Carregando dados de preço de: C:\Users\leona\OneDrive\Área de Trabalho\Machine-Learning---Stock-Prediction\CodigoExperimentos\ExperimentoFeatures\dados_preco\BEEF3_SA_historico.csv
Colunas originais: ['Close']
Features identificadas para BEEF3.SA: ['Close_Feature']
Target identificado para BEEF3.SA: Close_Target
Dimensões finais do DataFrame para BEEF3.SA: (1243, 2)

--- Iniciando Experimento: BEEF3.SA, Janela 1 ---
Epoch 1/200
